In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import joblib
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import statsmodels.api as sm
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    root_mean_squared_error, 
    r2_score, 
    max_error, 
    explained_variance_score
)

def dummy_plot(df_pred, target, unit, path_to_save, max_value, min_value, x_label, y_label, adj=True):
    """
    Generates regression analysis plots including scatter plots, histograms,
    and saves them as PNG files. Also generates a box-and-whisker plot
    for prediction errors.

    Parameters
    ----------
    df_pred : pd.DataFrame
        DataFrame containing columns: ['act_target', 'pred_target', 'adj_pred_target']
    target : str
        Target variable name for labeling saved plots.
    unit : str
        Unit of measurement for errors (e.g., "kg", "ms").
    path_to_save : str
        Directory path where plots will be saved.
    max_value : float
        Maximum value for axis limits.
    min_value : float
        Minimum value for axis limits.
    x_label : str
        Label for X-axis.
    y_label : str
        Label for Y-axis.
    adj : bool, optional
        If True, use adjusted predictions ('adj_pred_target'), otherwise use raw predictions ('pred_target').
    """

    # ---------------------------
    # Define Variables
    # ---------------------------
    y_test = df_pred['act_target']
    y_pred_adj = df_pred['adj_pred_target']
    y_pred = df_pred['pred_target']

    # Choose which predictions to plot
    Y = y_pred_adj if adj else y_pred
    X = y_test

    # ---------------------------
    # Regression Statistics
    # ---------------------------
    results = sm.OLS(Y, sm.add_constant(X)).fit()
    print(results.summary())

    # ---------------------------
    # Update Matplotlib Parameters
    # ---------------------------
    plt.rcParams.update({'font.size': 24, 
                        'figure.figsize': (10, 10), 
                        'figure.dpi': 100, 
                        'font.sans-serif': 'Arial',
                        'axes.linewidth': 2.0,
                        'lines.linewidth': 3.0,
                        'xtick.major.width': 2.0,
                        'xtick.minor.visible': True,
                        'xtick.minor.width': 2,
                        'xtick.top': True,
                        'xtick.direction': 'in',
                        'xtick.major.size': 10.0,
                        'xtick.minor.size': 4.0,
                        'xtick.minor.ndivs': 2,
                        'ytick.major.width': 2.0,
                        'ytick.minor.visible': True,
                        'ytick.minor.width': 2,
                        'ytick.right': True,
                        'ytick.direction': 'in',
                        'ytick.major.size': 10.0,
                        'ytick.minor.size': 4.0,
                        'ytick.minor.ndivs': 2
                        })

    # # Turn on top and right ticks for all plots
    # plt.rcParams['xtick.top'] = True
    # plt.rcParams['ytick.right'] = True

    # ---------------------------
    # Main Figure Layout
    # ---------------------------
    fig = plt.figure(figsize=(10, 10))
    gs = GridSpec(4, 4)

    ax_scatter = fig.add_subplot(gs[1:4, 0:3])  # scatter plot
    ax_hist_y = fig.add_subplot(gs[0, 0:3])     # histogram of actual
    ax_hist_x = fig.add_subplot(gs[1:4, 3])     # histogram of predicted

    fontsize = 20

    # ---------------------------
    # Scatter Plot (Predicted vs Actual)
    # ---------------------------
    ax_scatter.plot(X, Y, 'o', markersize=6, color='black', alpha=0.1)
    
    # Line of best fit
    offset = 0.00
    linear_fit = np.linspace(min_value-offset, max_value-offset, 100)
    ax_scatter.plot(
        linear_fit, 
        linear_fit*results.params[1] + results.params[0], 
        linewidth=2, color='blue', alpha=1.0, label='Line of Best Fit'
    )

    # Ideal y = x line
    y = x = np.linspace(min_value-offset, max_value-offset, 100)
    ax_scatter.plot(x, y, '--', linewidth=2, color='red', alpha=1.0, label='Ideal Regressor')
    ax_scatter.legend(loc='upper left', fontsize=fontsize)

    # Axes formatting
    ticks = np.linspace(min_value, max_value, 6)
    ax_scatter.set_xlabel(x_label, fontsize=fontsize)
    ax_scatter.set_ylabel(y_label, fontsize=fontsize)
    ax_scatter.tick_params(axis='both', labelsize=fontsize, direction="in")
    ax_scatter.set_xlim(min_value-0.5, max_value+0.5)
    ax_scatter.set_ylim(min_value-0.5, max_value+0.5)
    ax_scatter.set_xticks(ticks)
    ax_scatter.set_yticks(ticks)

    # ---------------------------
    # Histograms
    # ---------------------------
    bins = np.linspace(min_value-0.5, max_value+0.5, 51)
    color = 'skyblue'

    # Histogram of actual values
    ax_hist_x.hist(Y, bins=bins, orientation='horizontal', color=color, histtype='bar', alpha=1.0, edgecolor='white')
    ax_hist_x.set_xlabel('Count', fontsize=fontsize)
    ax_hist_x.tick_params(axis='both', labelsize=fontsize, direction="in", labelleft=False)
    ax_hist_x.set_xticks([0, 100, 200])    
    ax_hist_x.set_yticks(ticks)
    ax_hist_x.set_ylim(min_value-0.5, max_value+0.5)

    # Histogram of predicted values
    ax_hist_y.hist(y_test, bins=bins, color=color, histtype='bar', alpha=1.0, edgecolor='white')
    ax_hist_y.set_ylabel('Count', fontsize=fontsize)
    ax_hist_y.tick_params(axis='both', labelsize=fontsize, direction="in", labelbottom=False)
    ax_hist_y.set_xticks(ticks)
    ax_hist_y.set_yticks([0, 100, 200])
    ax_hist_y.set_xlim(min_value-0.5, max_value+0.5)

    # Label histogram plot area for GBFS and DFT
    ax_hist_x.text(30, max_value*0.9, 'GBFS', fontsize=fontsize, color='black', rotation=270)
    ax_hist_y.text(max_value*0.9, 30, 'DFT', fontsize=fontsize, color='black') # 'Expt'
    

    # ---------------------------
    # Statistics Annotations
    # ---------------------------
    font_black = {'family': 'Arial','color': 'black','size': fontsize-3}
    font_red   = {'family': 'Arial','color': 'red','size': fontsize-3}
    font_blue  = {'family': 'Arial','color': 'blue','size': fontsize-3}

    ax_scatter.text(max_value*0.42, max_value*0.55, r'$y = x$', fontdict=font_red)
    ax_scatter.text(
        max_value*0.55, max_value*0.45, 
        r'$y = {:.3f}x + {:.3f}$'.format(results.params[1], results.params[0]), 
        fontdict=font_blue
    )

    xx = max_value*0.95
    ax_scatter.text(xx, 2.0,  fr'$N$ = {len(X)}', fontdict=font_black, ha='right')
    ax_scatter.text(xx, 1.5,  fr'$R^2$ = {r2_score(X, Y):.3f}', fontdict=font_black, ha='right')
    ax_scatter.text(xx, 1.0,  fr'$MAE$ = {mean_absolute_error(X, Y):.3f} {unit}', fontdict=font_black, ha='right')
    ax_scatter.text(xx, 0.5,  fr'$RMSE$ = {root_mean_squared_error(X, Y):.3f} {unit}', fontdict=font_black, ha='right')
    ax_scatter.text(xx, 0.0,  fr'$MSE$ = {mean_squared_error(X, Y):.3f} {unit}$^2$', fontdict=font_black, ha='right')


    # ---------------------------
    # Save Main Regression Plot
    # ---------------------------
    fig.savefig(f"{path_to_save}/regression_filtered_{target}.png", dpi=500, bbox_inches="tight")
    fig.savefig(f"{path_to_save}/regression_filtered_{target}.pdf", bbox_inches="tight")
    print('Saved:', f"regression_filtered_{target}.png")
    plt.show()

    # # ---------------------------
    # # Box-and-Whisker Plot (Prediction Errors)
    # # ---------------------------
    # errors = Y - X  # prediction residuals
    # plt.figure(figsize=(6, 8))
    # plt.boxplot(errors, vert=True, patch_artist=True, boxprops=dict(facecolor='skyblue'))
    # plt.title(f'Prediction Errors for {target}', fontsize=16)
    # plt.ylabel(f'Error ({unit})', fontsize=14)

    # # Save Boxplot
    # plt.savefig(f"{path_to_save}/boxplot_errors_{target}.png", dpi=500, bbox_inches="tight")
    # print('Saved:', f"boxplot_errors_{target}.png")
    # plt.show()

In [ ]:
target = 'bandgap' 
unit = 'eV'

cwd = os.getcwd()
file_path = f"{cwd}\\pkl\\{target}_results"

output_file_path = f"{file_path}\\final" # \\expt"

file_name = '2d_combined' # 'expt'

df_pred = joblib.load(f"{file_path}\\final\\{file_name}_{target}_pred.pkl") # \\expt
print(df_pred.head())


In [ ]:
# retrieve the original data set
df_data = joblib.load(f"{file_path}\\..\\{file_name}_data.pkl")
print(df_data.head())

In [ ]:
# select only those entries from df_data and df_pred where the value of'task_id' column in df_pred matches the row index in df_data
# df_merged = df_data.loc[df_data.index.isin(df_pred['task_id'])]

# df_merged = df_data.merge(
#     df_pred,
#     left_index=True,
#     right_on='task_id',
#     how='inner'
# )

df_merged = (
    df_data
    .loc[df_data.index.isin(df_pred['task_id'])]
    .join(df_pred.set_index('task_id'), how='inner')
    .assign(task_id=lambda x: x.index)
)

print(df_merged.shape)
print(df_merged.head())

In [ ]:
# filter df_merged to only include rows where 'is_metal' is False and 'is_stable' is True
df_filtered = df_merged # df_merged[(df_merged['is_metal'] == False) & (df_merged['is_stable'] == True)]


# # FOR EXPT: rename columns of df_pred to match the pred_target, adj_pred_target (copy from pred_target), residual_error and task_id format needed for dummy_plot
# df_filtered = df_pred.rename(columns={
#     'expt_bandgap': 'act_target',
#     'gbfs_pred': 'pred_target'
# })
# df_filtered['adj_pred_target'] = df_filtered['pred_target']  # create adj_pred_target as a copy of pred_target for now
# df_filtered['residual_error'] = df_filtered['act_target'] - df_filtered['pred_target']
# df_filtered['task_id'] = df_filtered.index

print(df_filtered.head())

joblib.dump(df_filtered, f"{output_file_path}\\{file_name}_{target}_filtered_pred.pkl")

In [ ]:
dummy_plot(
            df_pred=df_filtered, 
            target=target, 
            unit=unit, 
            path_to_save = output_file_path, 
            max_value = 2, 
            min_value = -5, 
            x_label = f"DFT Prediction of log(Band Gap (${unit}$))", # "Experimental Value of Band Gap [${unit}$]"
            y_label = f"GBFS Prediction of log(Band Gap [${unit}$])", # "GBFS Prediction of {target} (${unit}$)",
            adj=False
            )

In [ ]:
import seaborn as sns

# Generate residual plot
# Calculate residual error
df_filtered['residual_error'] = df_filtered['act_target'] - df_filtered['adj_pred_target']

residual_summary = df_filtered['residual_error'].describe()
print("\nResidual Error Summary Statistics:")
print(residual_summary)
# residual_mean = round(residual_summary['mean'], 3)
# residual_std = round(residual_summary['std'], 3)

# Plot Histogram
plt.figure(figsize=(10,10))
fig, ax1 = plt.subplots()


# plot a kernel density estimate scaled to the existing axes
sns.kdeplot(df_filtered['residual_error'], color='skyblue', linewidth=2, ax=ax1, fill=True)
plt.xlabel('Residual Error [eV]')
#colour under the KDE line
# Add a label for the KDE plot
plt.text(residual_summary["mean"]+(residual_summary["std"]), 0.5, 'KDE', ha='left', va='center', color='skyblue')

ax2 = ax1.twinx()

sns.set(font_scale=2.5)
sns.set_style("ticks")
bins = np.arange(-6,2,0.1)
sns.histplot(df_filtered['residual_error'], bins=bins, alpha=1.0, color='skyblue',ax=ax2)
plt.xlabel('Residual Error [eV]')
plt.ylabel('Frequency')
plt.xlim([-6.0,2.0]) # ([-2.5, 2.5]) #
plt.text(1.8, 150, 'Residual Error:', ha = 'right', va = 'center')
plt.text(1.8, 100, f'Mean = {residual_summary["mean"]:.3f} {unit}', ha = 'right', va = 'center')
plt.text(1.8, 50, f'Std. Dev. = {residual_summary["std"]:.3f} {unit}', ha = 'right', va = 'center')

ax1.spines['top'].set_linewidth(3)
ax1.spines['bottom'].set_linewidth(3)
ax1.spines['left'].set_linewidth(3)
ax2.spines['right'].set_linewidth(3)

ax1.spines['top'].set_color('black')
ax1.spines['bottom'].set_color('black')
ax1.spines['left'].set_color('black')
ax2.spines['right'].set_color('black')

ax1.xaxis.set_ticks_position('both')
ax1.yaxis.set_ticks_position('left')
ax2.yaxis.set_ticks_position('right')


ax1.tick_params(axis='x', which='major', direction='in', width=3, length=10, color='black')
ax1.tick_params(axis='x', which='minor', direction='in', width=3, length=5, color='black')
ax2.tick_params(axis='y', which='major', direction='in', width=3, length=10, color='black')
ax2.tick_params(axis='y', which='minor', direction='in', width=3, length=5, color='black')
ax1.tick_params(axis='y', which='major', direction='in', width=3, length=10, color='black')
ax1.tick_params(axis='y', which='minor', direction='in', width=3, length=5, color='black')
# ax2.tick_params(axis='x', which='major', direction='in', width=3, length=10, color='black')
# ax2.tick_params(axis='x', which='minor', direction='in', width=3, length=5, color='black')

plt.savefig(f"{output_file_path}//{file_name}_{target}_filtered_residual_histogram.png", dpi = 500, bbox_inches="tight")
plt.savefig(f"{output_file_path}//{file_name}_{target}_filtered_residual_histogram.pdf", bbox_inches="tight")

In [ ]:
# Combine the already generated 2x2 plot and the SHAP plot for selected features
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(30, 20))
gs = gridspec.GridSpec(1, 2, width_ratios=[2, 1.5])

# Load images
img_combined = mpimg.imread(f'{output_file_path}//regression_filtered_{target}.png')
# img_shap = mpimg.imread(f'{output_file_path}//{target}_SHAP_top_{no_of_features_to_plot}_features_plot.png')
img_shap = mpimg.imread(f'{output_file_path}//{file_name}_{target}_filtered_residual_histogram.png')

# (a-d) Combined 2x2 plot
ax = fig.add_subplot(gs[0, 0])
ax.imshow(img_combined)
ax.axis('off')
ax.set_title('(a)', loc='left', fontsize=36, fontweight='bold', x=0.0, y=0.95)

# (e) SHAP summary plot for selected features
ax = fig.add_subplot(gs[0, 1])
ax.imshow(img_shap)
ax.axis('off')
ax.set_title('(b)', loc='left', fontsize=36, fontweight='bold', x=0.0, y=0.95)

plt.tight_layout()
plt.subplots_adjust(wspace=0)  # Reduce horizontal spacing
# plt.savefig(f'{output_file_path}//{target}_all_training_composite.svg', format='svg')
plt.savefig(f'{output_file_path}//{file_name}_{target}_filtered_performance_composite.png', dpi=300)
plt.savefig(f'{output_file_path}//{file_name}_{target}_filtered_performance_composite.pdf', format='pdf')
plt.show()

In [ ]:
from pymatgen.core import Composition
from matplotlib.ticker import MultipleLocator
import math

# ---------------------------
# Update Matplotlib Parameters
# ---------------------------
plt.rcParams.update({'font.size': 20, 
                     'xtick.labelsize': 20,
                     'ytick.labelsize': 20,
                    'figure.figsize': (10, 10), 
                    'figure.dpi': 100, 
                    'font.sans-serif': 'Arial',
                    'axes.linewidth': 2.0,
                    'lines.linewidth': 3.0,
                    'xtick.major.width': 2.0,
                    'xtick.minor.visible': True,
                    'xtick.minor.width': 2,
                    'xtick.top': True,
                    'xtick.direction': 'in',
                    'xtick.major.size': 10.0,
                    'xtick.minor.size': 4.0,
                    'xtick.minor.ndivs': 2,
                    'ytick.major.width': 2.0,
                    'ytick.minor.visible': True,
                    'ytick.minor.width': 2,
                    'ytick.right': True,
                    'ytick.direction': 'in',
                    'ytick.major.size': 10.0,
                    'ytick.minor.size': 4.0,
                    'ytick.minor.ndivs': 2
                    })

# define chemical families and their colours
chemical_families = {
    'Oxide': {'elements': ['O'], 'color': 'red', 'marker': 'o'},
    'Phosphide': {'elements': ['P'], 'color': 'purple', 'marker': 'D'},
    'Halide': {'elements': ['F', 'Cl', 'Br', 'I'], 'color': 'green', 'marker': 's'},
    'Nitride': {'elements': ['N'], 'color': 'orange', 'marker': '^'},
    'Chalcogenide': {'elements': ['S', 'Se', 'Te'], 'color': 'blue', 'marker': 'v'},
    'Other': {'elements': [], 'color': 'grey', 'marker': 'X'}
}
# function to determine chemical family based on formula using pymatgen Composition to parse the formula and check for the presence of elements in each family
# allow for multiple families to be detected
def determine_family(formula):
    comp = Composition(formula)
    elements = [el.symbol for el in comp.elements]
    families = []
    for family, data in chemical_families.items():
        if any(el in data['elements'] for el in elements):
            families.append(family)
    return families if families else ['Others']


df_filtered['chemical_family'] = df_filtered['formula'].apply(determine_family)

In [ ]:
# plot each chemical family separately in its own subplot of a 2x3 grid
fig, axs = plt.subplots(2, 3, figsize=(20, 15))
for ax, (family, data) in zip(axs.flatten(), chemical_families.items()):
    # Plot all materials belonging to this family, including those in multiple families
    for i, row in df_filtered.iterrows():
        families = row['chemical_family']
        if not isinstance(families, list):
            families = [families]
        if family in families:
            ax.scatter(row['bandgap'], row['pred_target'], color=data['color'], label=family, s=100, marker=data['marker'], alpha=0.3, edgecolors=data['color'], linewidths=0.5)
    # Prepare subset for stats and fit
    subset = df_filtered[df_filtered['chemical_family'].apply(lambda fams: family in fams if isinstance(fams, list) else fams == family)]
    max_val =  math.ceil(max(subset['bandgap'].max(), subset['pred_target'].max()) if not subset.empty else 7)
    ax.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='y=x Line')
    ax.set_xlabel('DFT Prediction of Band Gap [eV]', fontsize=20)
    ax.set_ylabel('GBFS Prediction of Band Gap [eV]', fontsize=20)
    ax.set_xlim(0, max_val+0.5)
    ax.set_ylim(0, max_val+0.5)
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.set_minor_locator(MultipleLocator(0.5))
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_minor_locator(MultipleLocator(0.5))
    ax.set_title(f'{family}', fontsize=24)
    if len(subset) > 5:
        m, b = np.polyfit(subset['bandgap'], subset['pred_target'], 1)
        ax.plot(subset['bandgap'], m*subset['bandgap'] + b, color='black', linestyle='-', label='Lin. Fit')
        ax.text(max_val*0.05, max_val, f'R²: {r2_score(subset["bandgap"], subset["pred_target"]):.3f}')
        ax.text(max_val*0.05, max_val*0.94, f'RMSE: {root_mean_squared_error(subset["bandgap"], subset["pred_target"]):.3f} eV')
        ax.text(max_val*0.05, max_val*0.88, f'MAE: {mean_absolute_error(subset["bandgap"], subset["pred_target"]):.3f} eV')
        ax.text(max_val*0.05, max_val*0.82, f'N: {len(subset)}')
        ax.text(max_val*0.05, max_val*0.76, f'Lin. Fit: y={m:.2f}x+{b:.2f}')
    else:
        ax.text(max_val*0.05, max_val, f'N: {len(subset)}')
    # ax.legend( by_label.values(), by_label.keys(), loc='lower right', fontsize=16)

# Adjust layout
plt.tight_layout()
#save figure as png and pdf
plt.savefig(f"{output_file_path}\\dft_vs_gbfs_bandgap_by_chemical_family_subplots.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{output_file_path}\\dft_vs_gbfs_bandgap_by_chemical_family_subplots.pdf", dpi=300, bbox_inches='tight')
plt.show()